# Tratamento de Dados
Este notebook aplica as mesmas transformações do `tratativa.ipynb`, mas opera apenas em DataFrames na memória.
As funções não precisam de caminhos de arquivo e são ideais para uso no Google Colab.

In [ ]:
import pandas as pd
import re

# As funções abaixo operam diretamente sobre DataFrames em memória.
# Ideal para uso em Google Colab sem referências a arquivos locais.

In [ ]:
def limpar_avaliacoes(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip()
    texto = re.sub(r'^[\W_]+', '', texto, flags=re.UNICODE)
    texto = re.sub(r'\s{2,}', ' ', texto)

    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)
    texto = re.sub(r'([.,;:!?])\1+', r'\1', texto)
    texto = re.sub(r'([.,;:!?])(?=[^\s])', r'\1 ', texto)

    mapeamento = {
        r'\b(n[ãa]o|nã|nao)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ''
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break

        if original.isupper():
            return alvo.upper()
        if original and original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    texto = re.sub(r'(\.\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)
    texto = re.sub(r'[\s]+$', '', texto)
    return texto

In [ ]:
def juntar_titulo_mensagem(df, coluna_titulo='review_comment_title', coluna_mensagem='review_comment_message'):
    if coluna_titulo not in df.columns or coluna_mensagem not in df.columns:
        print(f'Erro: colunas {coluna_titulo} ou {coluna_mensagem} não encontradas.')
        return df

    df[coluna_titulo] = df[coluna_titulo].astype('string')
    df[coluna_mensagem] = df[coluna_mensagem].astype('string')

    def combinar(titulo, mensagem):
        titulo = str(titulo).strip() if pd.notna(titulo) else ''
        mensagem = str(mensagem).strip() if pd.notna(mensagem) else ''

        if titulo and mensagem:
            return f'{titulo} - {mensagem}'
        return titulo or mensagem

    df[coluna_mensagem] = df.apply(lambda row: combinar(row[coluna_titulo], row[coluna_mensagem]), axis=1)
    return df

In [ ]:
def processar_dataframe(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    df[nome_coluna] = df[nome_coluna].astype('string').apply(limpar_avaliacoes)
    return df

In [ ]:
def remover_linhas_sem_review(df, coluna='review_comment_message'):
    if coluna not in df.columns:
        print(f'Erro: A coluna {coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    mask = df[coluna].astype('string').str.strip() == ''
    removidas = int(mask.sum())
    df = df.drop(df[mask].index).reset_index(drop=True)
    print(f'Removidas {removidas} linhas sem valor em {coluna}.')
    return df

In [ ]:
def apagar_coluna(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    df = df.drop(columns=[nome_coluna])
    return df

In [ ]:
def converter_para_string(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
        return df

    df = df.copy()
    df[nome_coluna] = df[nome_coluna].astype('string')
    print(f"Coluna '{nome_coluna}' convertida para string.")
    return df


def converter_para_datetime(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
        return df

    df = df.copy()
    df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
    print(f"Coluna '{nome_coluna}' convertida para datetime.")
    return df

In [ ]:
def converter_datas(df):
    """
    Converte colunas de data para tipo datetime.
    Útil para colunas de timestamp e datas em geral.
    """
    df = df.copy()
    
    # Colunas de data comuns em avaliacoes
    colunas_data_avaliacoes = ['review_answer_timestamp', 'review_creation_date']
    for col in colunas_data_avaliacoes:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Coluna '{col}' convertida para datetime.")
    
    # Colunas de data comuns em pedidos
    colunas_data_pedidos = ['order_delivered_carrier_date', 'order_approved_at', 
                            'order_estimated_delivery_date', 'order_purchase_timestamp', 
                            'order_delivered_customer_date']
    for col in colunas_data_pedidos:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Coluna '{col}' convertida para datetime.")
    
    # Colunas de data comuns em itens
    if 'shipping_limit_date' in df.columns:
        df['shipping_limit_date'] = pd.to_datetime(df['shipping_limit_date'], errors='coerce')
        print(f"Coluna 'shipping_limit_date' convertida para datetime.")
    
    return df


In [ ]:
def converter_categorias(df):
    """
    Converte colunas categóricas para tipo 'category' para economizar memória.
    Inclui cidades, estados, categorias de produto, tipos de pagamento e status de pedido.
    """
    df = df.copy()
    
    # Cidades e Estados em clientes
    if 'customer_city' in df.columns:
        df['customer_city'] = df['customer_city'].astype('category')
        print("Coluna 'customer_city' convertida para category.")
    if 'customer_state' in df.columns:
        df['customer_state'] = df['customer_state'].astype('category')
        print("Coluna 'customer_state' convertida para category.")
    
    # Categoria de produtos
    if 'product_category_name' in df.columns:
        df['product_category_name'] = df['product_category_name'].astype('category')
        print("Coluna 'product_category_name' convertida para category.")
    
    # Cidades e Estados em vendedores
    if 'seller_city' in df.columns:
        df['seller_city'] = df['seller_city'].astype('category')
        print("Coluna 'seller_city' convertida para category.")
    if 'seller_state' in df.columns:
        df['seller_state'] = df['seller_state'].astype('category')
        print("Coluna 'seller_state' convertida para category.")
    
    # Tipo de pagamento
    if 'payment_type' in df.columns:
        df['payment_type'] = df['payment_type'].astype('category')
        print("Coluna 'payment_type' convertida para category.")
    
    # Geolocalização
    if 'geolocation_city' in df.columns:
        df['geolocation_city'] = df['geolocation_city'].astype('category')
        print("Coluna 'geolocation_city' convertida para category.")
    if 'geolocation_state' in df.columns:
        df['geolocation_state'] = df['geolocation_state'].astype('category')
        print("Coluna 'geolocation_state' convertida para category.")
    
    # Status do pedido
    if 'order_status' in df.columns:
        df['order_status'] = df['order_status'].astype('category')
        print("Coluna 'order_status' convertida para category.")
    
    return df


In [ ]:
# Exemplo de uso em memória no Colab:
df = pd.read_csv('/content/avaliacoes.csv')  # ajuste o caminho conforme o arquivo no Colab

df = processar_dataframe(df, 'review_comment_title')
df = processar_dataframe(df, 'review_comment_message')
df = juntar_titulo_mensagem(df)
df = remover_linhas_sem_review(df, 'review_comment_message')

df = df.reset_index(drop=True)

# Exibir o DataFrame tratado sem gravar o arquivo original
print(df.head())

In [ ]:
# Exemplo completo com tratamento de tipos de variáveis:
# Para avaliacoes
df_avaliacoes = pd.read_csv('/content/avaliacoes.csv')
df_avaliacoes = processar_dataframe(df_avaliacoes, 'review_comment_title')
df_avaliacoes = processar_dataframe(df_avaliacoes, 'review_comment_message')
df_avaliacoes = juntar_titulo_mensagem(df_avaliacoes)
df_avaliacoes = remover_linhas_sem_review(df_avaliacoes, 'review_comment_message')
df_avaliacoes = converter_para_string(df_avaliacoes, 'review_id')
df_avaliacoes = converter_datas(df_avaliacoes)
print(df_avaliacoes.head())

# Para outras tabelas (clientes, produtos, etc)
df_clientes = pd.read_csv('/content/clientes.csv')
df_clientes = converter_para_string(df_clientes, 'customer_id')
df_clientes = converter_categorias(df_clientes)
print(df_clientes.head())